In [ ]:
# Install required packages
!pip install fastapi nest-asyncio pyngrok uvicorn transformers pillow torch

# Server Code
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
from io import BytesIO
import torch
import uvicorn
from pyngrok import ngrok
import nest_asyncio
import os

In [ ]:
# Initialize
nest_asyncio.apply()

# Load model
processor = AutoProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = AutoModelForVision2Seq.from_pretrained("Salesforce/blip-image-captioning-large").to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# FastAPI app
app = FastAPI()

# CORS Configuration
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def health_check():
    return {"status": "ready"}

@app.post("/caption/")
async def generate_caption(file: UploadFile = File(...)):
    try:
        # Validate file
        if not file.content_type.startswith("image/"):
            raise HTTPException(400, detail="Invalid image format")

        # Read and process image
        image_bytes = await file.read()
        image = Image.open(BytesIO(image_bytes)).convert("RGB")

        # Generate caption
        inputs = processor(images=image, return_tensors="pt").to(model.device)
        generated_ids = model.generate(**inputs, max_new_tokens=50)
        caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        return {"caption": caption}

    except Exception as e:
        raise HTTPException(500, detail=str(e))



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
# Ngrok tunnel
ngrok.set_auth_token("2u7UhANCJAZuagJlAHuuCDWYkyy_7a6aTys1kS3s7GjLGu2P2")
public_url = ngrok.connect(8000, bind_tls=True).public_url
print(f"Public URL: {public_url}")

# Start server
uvicorn.run(app, host="0.0.0.0", port=8000)

INFO:     Started server process [15158]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: https://fe62-34-139-49-186.ngrok-free.app
INFO:     160.20.123.9:0 - "GET / HTTP/1.1" 200 OK
INFO:     160.20.123.9:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     152.58.239.224:0 - "POST /caption/ HTTP/1.1" 200 OK
INFO:     152.58.239.19:0 - "POST /caption/ HTTP/1.1" 200 OK
INFO:     152.58.239.74:0 - "POST /caption/ HTTP/1.1" 200 OK
INFO:     152.58.239.74:0 - "POST /caption/ HTTP/1.1" 200 OK
INFO:     152.58.239.227:0 - "POST /caption/ HTTP/1.1" 200 OK
INFO:     152.58.239.159:0 - "POST /caption/ HTTP/1.1" 200 OK
INFO:     152.58.239.20:0 - "POST /caption/ HTTP/1.1" 200 OK
INFO:     152.58.239.115:0 - "POST /caption/ HTTP/1.1" 200 OK
INFO:     152.58.239.150:0 - "POST /caption/ HTTP/1.1" 200 OK
INFO:     152.58.239.150:0 - "POST /caption/ HTTP/1.1" 200 OK
INFO:     152.58.239.150:0 - "POST /caption/ HTTP/1.1" 200 OK
INFO:     152.58.239.54:0 - "POST /caption/ HTTP/1.1" 200 OK
INFO:     152.58.239.7:0 - "POST /caption/ HTTP/1.1" 200 OK
INFO:     152.58.239.111:0 